# Interpretability vs Complexity

**Topic:** The Explainability-Performance Tradeoff

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** where each major algorithm sits on the interpretability-complexity spectrum
- **Explain** why some industries legally require interpretable models regardless of performance
- **Interpret** the business consequences of choosing a complex model over an interpretable one

> **Tip:** Start with **Any domain** and hover over each algorithm point to see its plain English description. Then select **Healthcare** and notice which algorithms disappear from consideration — not because of performance, but because of regulatory requirements.

---
## How we got here

In **12_model_complexity_and_capacity.ipynb** we saw that more capacity does not always mean better generalization. Now we ask a different question: even when a complex model is more accurate, should you use it?

This connects directly to:
- **math/statistics_for_ml/04_regularization_mathematics.ipynb** — regularization is one tool for managing the complexity side of this tradeoff
- **preprocessing/09_pipelines.ipynb** — pipelines make the entire modeling workflow reproducible and auditable, which is essential when interpretability matters

---
## Why this matters for data science

A model that is 2% more accurate but completely uninterpretable may be the wrong choice when a doctor needs to explain a diagnosis or a bank needs to justify a loan rejection.

In many industries, interpretability is not optional — it is a legal requirement. The EU's GDPR grants citizens a "right to explanation" for automated decisions. The US Fair Housing Act requires lenders to explain adverse credit decisions. A black-box model may be technically excellent and legally unusable.

---
## Try it yourself

In [ ]:
ALGORITHMS = [
    ("Linear Regression",   0.05, 0.98, "Finance, science, economics"),
    ("Logistic Regression", 0.10, 0.95, "Healthcare, credit scoring, HR"),
    ("Decision Tree",       0.30, 0.80, "Healthcare, insurance, legal"),
    ("KNN",                 0.40, 0.65, "Recommendation systems"),
    ("Random Forest",       0.65, 0.45, "Fraud detection, genomics"),
    ("Gradient Boosting",   0.80, 0.25, "Competitions, ad targeting"),
    ("Neural Network",      0.98, 0.05, "Vision, NLP, drug discovery"),
]

INDUSTRIES = {
    "Healthcare": ["Linear Regression", "Logistic Regression", "Decision Tree"],
    "Finance / banking": ["Linear Regression", "Logistic Regression", "Decision Tree"],
    "E-commerce": ["Random Forest", "Gradient Boosting", "KNN"],
    "Autonomous vehicles": ["Neural Network", "Gradient Boosting"],
    "Any domain": [a[0] for a in ALGORITHMS],
}

out = Output()

industry_dd = Dropdown(
    options=list(INDUSTRIES.keys()),
    value="Any domain",
    description="Industry:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="380px"))

def render(industry):
    highlight = INDUSTRIES[industry]
    traces = []
    for name, complexity, interpretability, use_case in ALGORITHMS:
        is_hl = name in highlight
        traces.append(go.Scatter(
            x=[complexity], y=[interpretability],
            mode="markers+text",
            marker=dict(
                size=18 if is_hl else 12,
                color=PALETTE["primary"] if is_hl else PALETTE["muted"],
                opacity=1.0 if is_hl else 0.4,
                line=dict(color="white", width=2 if is_hl else 0)),
            text=[name], textposition="top center",
            textfont=dict(size=11 if is_hl else 9,
                          color=PALETTE["primary"] if is_hl else PALETTE["muted"],
                          family=FONT["family"]),
            name=name,
            hovertemplate=f"<b>{name}</b><br>Use case: {use_case}<extra></extra>",
        ))
    layout = base_layout(
        title=f"Interpretability vs Complexity spectrum — {industry}",
        xaxis_title="Complexity → (low to high)",
        yaxis_title="Interpretability → (low to high)")
    layout.update(xaxis=dict(range=[-0.05, 1.1], tickvals=[]),
                  yaxis=dict(range=[-0.05, 1.1], tickvals=[]),
                  showlegend=False, height=450)
    with out:
        clear_output(wait=True)
        fig = go.Figure(data=traces, layout=layout)
        display(go.FigureWidget(fig))

def on_change(change): render(industry_dd.value)
industry_dd.observe(on_change, names="value")
display(VBox([industry_dd, out]))
render("Any domain")

---
## What's happening?

Think of a simple recipe versus a restaurant with 20 chefs. The restaurant produces better food, but you cannot replicate it at home or explain exactly how they made it. A simple recipe is less impressive but completely transparent.

The same tradeoff applies to ML algorithms:

**Fully interpretable:** Linear regression, logistic regression — every coefficient has a direct, plain-English meaning. "A one-unit increase in age is associated with a 2.3% higher risk of churn."

**Partially interpretable:** Decision trees, random forests — individual trees are readable, but the ensemble of 100 trees is not.

**Black box:** Neural networks, gradient boosting — predictions are accurate but the internal computation is not human-readable.

> "A model that is 2% more accurate but completely uninterpretable may be the wrong choice when a doctor needs to explain a diagnosis or a bank needs to justify a loan rejection."

**Discussion question:** You are building a model to predict which patients need urgent care. Your neural network gets 94% accuracy; your decision tree gets 89%. Which do you deploy and why?

| Algorithm | Complexity | Interpretability | Regulated industry appropriate? | When to choose it |
|---|---|---|---|---|
| Linear Regression | Very low | Full | Yes | Baselines, science, economics |
| Logistic Regression | Very low | Full | Yes | Credit scoring, healthcare |
| Decision Tree | Low | High | Yes | When rules must be audited |
| KNN | Medium | Moderate | Sometimes | Recommendation, small data |
| Random Forest | High | Low | Partially | When accuracy matters more |
| Gradient Boosting | High | Very low | Rarely | Competitions, ad targeting |
| Neural Network | Very high | None | Rarely | Vision, NLP, audio |

---
## Real-world example: GDPR and the right to explanation

The EU's GDPR (Article 22) gives citizens the right to an explanation for automated decisions that significantly affect them — loan approvals, job screenings, medical recommendations. A neural network that outputs "rejected" cannot satisfy this requirement.

A logistic regression can: "Your application was declined primarily because your debt-to-income ratio (45%) exceeds our threshold (40%), and your credit history length (8 months) is below the median for approved applicants (36 months)."

In healthcare, financial services, and government contexts, choose the most interpretable model that meets the performance threshold — not the most accurate model.

---
## Key takeaway

> **Interpretability is not a nice-to-have — in regulated industries it is a legal requirement, and choosing a black-box model when stakeholders need explanations is a project risk, not just a design preference.**

---
*Next up: Parametric vs Nonparametric Models — how assumptions about data shape affect what models can and cannot learn*